# 31. 현재 운영 청킹·새 검색 인덱스

OCR 통과 가정. 원본은 읽기 전용이며 현재 Upstage 계층을 수정하지 않습니다. Python 실행은 conda skn25만 허용합니다.

## 재현과 책임

장시간 API·모델 실행은 동일 payload 캐시·체크포인트·실패 기록을 가진 runner가 담당합니다. 이 노트북의 셀은 저장된 실제 결과를 읽고 보여주며 보고서를 생성합니다. 유료 작업을 셀에서 처음부터 재실행하지 않습니다.

아래 각 Python 명령 앞에 `PYTHONDONTWRITEBYTECODE=1 conda run --no-capture-output -n skn25`를 붙입니다. 원본 OCR lane·코드, 동결된 JSON/SQLite·청크·새 인덱스, API 캐시와 입력 hash가 필요합니다. 입력 hash가 바뀌면 캐시를 자동 덮어쓰지 않고 중단합니다. 캐시가 없는 API 실행에는 키·네트워크·비용이 필요합니다. 원본 데이터와 API 키는 Git 커밋에 포함하지 않습니다. DB 쓰기는 database-manager가 담당합니다.

```bash
python -B notebooks/scripts/31_operational_chunking.py
# 새 FTS/Chroma DB 쓰기는 database-manager가 아래 모듈 함수로 수행합니다.
# 31_operational_chunking.py: 각 profile의 build_fts(profile), build_vectors(profile)
```

모든 결과가 완료된 뒤 `30_summarize_experiments.py`, `30_execute_experiment_notebooks.py`, `30_render_experiment_reports.py`, `30_verify_preservation.py` 순서로 최종 산출물을 확인합니다.

In [1]:
from pathlib import Path
import importlib.util, sys, json
root=Path.cwd()
if root.name=='notebooks': root=root.parent
spec=importlib.util.spec_from_file_location('structured_common',root/'notebooks/scripts/30_structured_search_common.py')
C=importlib.util.module_from_spec(spec);sys.modules[spec.name]=C;spec.loader.exec_module(C)
assert Path(sys.prefix).name=='skn25'
print({'environment':'skn25','model':C.MODEL,'reasoning':C.REASONING,'originals':'read only'})

{'environment': 'skn25', 'model': 'gpt-6-luna', 'reasoning': 'medium', 'originals': 'read only'}


In [2]:
files=['chunking_summary.json']
for name in files:
    path=C.STAGES[31]/name
    if not path.exists(): raise RuntimeError('Required stage artifact not complete: '+name)
    data=C.read_json(path)
    if 'records' in data:
        print(name,{'records':len(data['records']),'status_counts':{s:sum(r['status']==s for r in data['records']) for s in ['ok','failed']}})
    elif 'queries' in data: print(name,{'queries':len(data['queries']),'sha256':data['sha256']})
    else: print(json.dumps(data,ensure_ascii=False,indent=2)[:15000])

{
  "assumption": "OCR passed; current imperfect heading hierarchy retained",
  "audits": [
    {
      "audit": {
        "ambiguous_fact_indices": [],
        "chunking_contract": "ocr-paragraph-label-window-v1",
        "facts": 43,
        "input_path": "data/rag/runtime/working/run_f2d29820c6dca642/documents/BC__BC_Baro_Clear_Plus/upstage/ocr.txt",
        "input_sha256": "4febdaaf3fbe0131320fb5a16bcf92084c17ec3c8e1f7a7df988079c2efac6b6",
        "level_counts": {
          "benefit": 2,
          "card": 1,
          "page": 2,
          "section": 8
        },
        "mapped_facts": 43,
        "merged_benefit_windows": 41,
        "mid_text_truncation_windows": 0,
        "neighbor_omission_windows": 0,
        "ocr_text_sha256": "4febdaaf3fbe0131320fb5a16bcf92084c17ec3c8e1f7a7df988079c2efac6b6",
        "oversized_core_windows": 2,
        "pages": 2,
        "unmapped_fact_indices": []
      },
      "chunks": 13,
      "document_id": "BC/BC_Baro_Clear_Plus",
      "profile"

## 해석 범위

source-relative raw OCR 필드 라벨과 새로운 개발/확증 cohort입니다. 독립 PDF 검증, 전수 정답 recall, 운영 채택을 뜻하지 않습니다. 프로필 비교는 청킹·재정렬·근거 구성 전체의 비교입니다. 완전한 재현은 입력 hash·원본 코드 hash·새 인덱스·저장 API 응답과 보고서의 실패 분모를 함께 확인해야 합니다.

In [3]:
spec=importlib.util.spec_from_file_location('experiment_reports',root/'notebooks/scripts/30_render_experiment_reports.py')
R=importlib.util.module_from_spec(spec);sys.modules[spec.name]=R;spec.loader.exec_module(R)
print(R.render(31).relative_to(root))

docs/report/31_operational_chunking_comparison_report.html


In [4]:
for profile in ['card_page_section_benefit','parent_child_bundle']:
    print(profile,json.dumps(C.read_json(C.STAGES[31]/profile/'index_summary.json'),ensure_ascii=False))

card_page_section_benefit {"chunks": 3183, "distance": "squared_l2", "embedding_shape": [3183, 1536], "index_bytes": 230298728, "model": "text-embedding-3-small", "profile": "card_page_section_benefit", "wall_seconds": 34.76262792399939}
parent_child_bundle {"chunks": 1834, "distance": "squared_l2", "embedding_shape": [1834, 1536], "index_bytes": 99005440, "model": "text-embedding-3-small", "profile": "parent_child_bundle", "wall_seconds": 21.76788689100067}
